In [1]:

import re
import random
import time
import torch
from tqdm.auto import tqdm
from IPython.display import display, Markdown
from transformers import GPT2LMHeadModel, GPT2Tokenizer

from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

from collections import Counter
from IPython.display import display, Markdown
import pandas as pd


model_name = "sberbank-ai/rugpt3large_based_on_gpt2"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tok = GPT2Tokenizer.from_pretrained(model_name)
# model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
# model.eval()

MAX_NEW_TOKENS = 1_000
TOP_P = 0.95
REPETITION_PENALTY = 2.0
NUM_BEAMS = 1
DO_SAMPLE = True
TOTAL_ATTEMPTS = 15_000
STATS_EVERY = 100
TOTAL_CUDAS = 4


In [2]:
assert torch.cuda.device_count() >= TOTAL_CUDAS, f"Нужно минимум {TOTAL_CUDAS} GPU"

models = []
for i in range(TOTAL_CUDAS):
    device = f"cuda:{i}"

    model_i = GPT2LMHeadModel.from_pretrained(
        model_name,
        torch_dtype=torch.float16
    ).to(device)

    model_i.eval()
    models.append(model_i)

    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

`torch_dtype` is deprecated! Use `dtype` instead!


GPU 0: NVIDIA RTX A5000
GPU 1: NVIDIA RTX A5000
GPU 2: NVIDIA RTX A5000
GPU 3: NVIDIA RTX A5000


In [ ]:
words = [
    'является', 'стало', 'было', 'стала', 'была',
    'становится', 'считается', 'остается', 'будет', 'стал'
]

prompts = [
    #1
    """Вопрос о роли современных технологий вызывает активные дискуссии среди учёных, 
    представителей бизнеса и общества. Одни участники обсуждения обращают внимание 
    на открывающиеся возможности, другие подчёркивают существующие ограничения. 
    При этом общепринятое мнение...""",
    
    #2
    """В архиве нашли рукопись неизвестного автора. В ней подробно
    описывались события нескольких лет, встречи с необычными людьми
    и результаты научных опытов. Приводим отрывок из этой рукописи:""",

    #3
    """Молодой исследователь отправился в далёкую экспедицию. В своём
    блокноте он фиксировал наблюдения, записывал имена участников,
    описывал открытия и неожиданные происшествия. Однажды вечером
    он сделал следующую запись:""",

    #4
    """Когда команда приступила к реализации проекта, 
    первоначальные планы существенно отличались от итоговых.
    Со временем отдельные решения пересматривались, новые подходы проходили проверку, а
    некоторые идеи получали неожиданное продолжение. В результате...""",

    #5
    """В современной научной литературе результаты данного исследования 
    рассматриваются как важный вклад в развитие новой области знаний. 
    Полученные данные позволяют сделать вывод о том, что…"""
]


In [4]:

def generate(
    model, tok, text,
    do_sample=True, max_length=100, repetition_penalty=5.0,
    top_k=5, top_p=0.95, temperature=1,
    num_beams=None, no_repeat_ngram_size=3
):
    input_ids = tok.encode(text, return_tensors="pt")

    with torch.no_grad():
        out = model.generate(
            input_ids,
            max_length=max_length,
            repetition_penalty=repetition_penalty,
            do_sample=do_sample,
            top_k=top_k,
            top_p=top_p,
            temperature=temperature,
            num_beams=num_beams,
            no_repeat_ngram_size=no_repeat_ngram_size,
            pad_token_id=tok.eos_token_id
        )

    return [tok.decode(ids, skip_special_tokens=True) for ids in out]
    
def find_progress_with_pos(text):
    pos = 0
    found = []

    for word in words:
        match = re.search(
            r'(?<!\w)' + re.escape(word) + r'(?!\w)',
            text[pos:],
            re.IGNORECASE
        )

        if match is None:
            return found, word

        pos += match.end()
        found.append(word)

    return found, None


def find_progress(text):
    normalized_text = text.casefold().replace("ё", "е")

    found = []
    missing = []

    for word in words:
        normalized_word = word.casefold().replace("ё", "е")

        pattern = r'(?<!\w)' + re.escape(normalized_word) + r'(?!\w)'

        if re.search(pattern, normalized_text):
            found.append(word)
        else:
            missing.append(word)

    return found, missing


def print_stats(stats):
    if not stats:
        return

    df = pd.DataFrame(stats)
    total = len(df)

    print(f"\n{'=' * 60}")
    print(f"СТАТИСТИКА: завершено {total} операций")
    print(f"{'=' * 60}")

    print(f"Среднее число найденных слов: "
          f"{df['found_count'].mean():.2f} / {len(words)}")

    print(f"Медиана: {df['found_count'].median():.0f} / {len(words)}")
    print(f"Лучший результат: {df['found_count'].max()} / {len(words)}")

    print("\nДоля генераций по количеству найденных слов:")
    distribution = (
        df["found_count"]
        .value_counts()
        .sort_index()
        .rename_axis("Найдено слов")
        .to_frame("Количество")
    )
    distribution["Доля, %"] = (
        distribution["Количество"] / total * 100
    ).round(1)
    display(distribution)

    print("\nЧастота каждого слова:")
    word_counts = Counter(
        word
        for found_list in df["found"]
        for word in found_list
    )

    word_stats = pd.DataFrame([
        {
            "Слово": word,
            "Найдено раз": word_counts[word],
            "Доля генераций, %": round(
                word_counts[word] / total * 100, 1
            ),
        }
        for word in words
    ]).sort_values("Доля генераций, %")

    display(word_stats)

    print("\nРезультаты по промптам:")
    prompt_stats = (
        df.groupby("prompt_id")
        .agg(
            Операций=("found_count", "size"),
            Среднее=("found_count", "mean"),
            Максимум=("found_count", "max"),
        )
        .round(2)
        .sort_values("Среднее", ascending=False)
    )
    display(prompt_stats)

    print("\nРезультаты по параметрам генерации:")
    
    params_stats = (
        df.groupby([
            "temperature",
            "top_k",
            "no_repeat_ngram_size"
        ])
        .agg(
            Операций=("found_count", "size"),
            Среднее=("found_count", "mean"),
            Медиана=("found_count", "median"),
            Максимум=("found_count", "max"),
            Успехов=("success", "sum"),
        )
        .round(2)
        .sort_values(
            ["Среднее", "Максимум"],
            ascending=False
        )
    )
    
    display(params_stats)


In [5]:
def run_attempt(attempt, gpu_id):
    model_i = models[gpu_id]
    prompt_id = random.randrange(len(prompts))
    prompt = prompts[prompt_id]

    temperature = random.choice([0.7, 0.8, 1.0, 1.2])
    top_k = random.choice([20, 50, 100])
    no_repeat_ngram_size = random.choice([0, 2])

    input_ids = tok.encode(
        prompt, return_tensors="pt"
    ).to(f"cuda:{gpu_id}")

    start = time.time()

    with torch.no_grad():
        output = model_i.generate(
            input_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            temperature=temperature,
            top_k=top_k,
            top_p=TOP_P,
            repetition_penalty=REPETITION_PENALTY,
            num_beams=NUM_BEAMS,
            no_repeat_ngram_size=no_repeat_ngram_size,
            pad_token_id=tok.eos_token_id
        )

    torch.cuda.synchronize(gpu_id)
    elapsed = time.time() - start

    generated = tok.decode(
        output[0, input_ids.shape[1]:],
        skip_special_tokens=True
    )

    found, missing = find_progress(generated)
    count = len(found)

    result = {
        "attempt": attempt,
        "gpu": gpu_id,
        "prompt_id": prompt_id,
        "prompt": prompt,
        "text": generated,
        "found": found,
        "missing": missing,
        "count": count,
        "temperature": temperature,
        "top_k": top_k,
        'no_repeat_ngram_size': no_repeat_ngram_size,
        "elapsed": elapsed,
        "success": count == len(words)
    }

    # Краткая информация о каждой попытке с найденными словами
    if found:
        print(
            f"\nПопытка {attempt} | GPU {gpu_id} | "
            f"{elapsed:.1f} сек | "
            f"Промпт №{prompt_id + 1} | "
            f"temperature={temperature}, top_k={top_k}",
            flush=True
        )
        
        # print("\nИсходный промпт:\n", prompt, flush=True)
        print(f"\nНайдено: {count}/10 — {' → '.join(found)}", flush=True)
        print(f"Ещё не найдены: {', '.join(missing) if missing else 'все найдены'}", flush=True)
        # print(f"Следующее ожидаемое слово: {missing or 'все найдены'}", flush=True)
        # print("\nСгенерированный текст:\n", generated, flush=True)
        
        

    # Полный текст при успехе
    if result["success"]:
        print("\n🎉 ВСЕ 10 СЛОВ НАЙДЕНЫ!", flush=True)
        print(f"Попытка: {attempt} | GPU: {gpu_id}", flush=True)
        print(f"Параметры: temperature={temperature}, top_k={top_k}")
        print("\nПромпт:\n", prompt, flush=True)
        print("\nСгенерированный текст:\n", generated, flush=True)

    return result


In [ ]:
attempts_stats = []

best_count = 0
best_text = ""
best_prompt = ""
best_params = {}
success = False
start_all = time.time()

gpu_count = len(models)
print("gpu_count:", gpu_count)

with ThreadPoolExecutor(max_workers=gpu_count) as executor:
    for batch_start in tqdm(
        range(1, TOTAL_ATTEMPTS + 1, gpu_count),
        desc="Перебор"
    ):
        batch = [
            (attempt,executor.submit(run_attempt, attempt, gpu_id))
            for gpu_id in range(gpu_count)
            if (attempt := batch_start + gpu_id) <= TOTAL_ATTEMPTS
        ]

        for attempt, future in batch:
            result = future.result()

            attempts_stats.append({
                "attempt": result["attempt"],
                "gpu": result["gpu"],
                "prompt_id": result["prompt_id"] + 1,
                "temperature": result["temperature"],
                "top_k": result["top_k"],
                'no_repeat_ngram_size': result["no_repeat_ngram_size"],
                "found_count": result["count"],                
                "found": result["found"].copy(),
                "missing": result["missing"].copy(),
                "elapsed": result["elapsed"],
                "success": result["success"],
            })

            if len(attempts_stats) % STATS_EVERY == 0:
                print_stats(attempts_stats)

            if result["count"] > best_count:
                best_count = result["count"]
                best_text = result["text"]
                best_prompt = result["prompt"]
                best_params = {
                    "temperature": result["temperature"],
                    "top_k": result["top_k"]
                }

                print("\n🏆 НОВЫЙ РЕКОРД!", flush=True)
                print(f"Попытка: {attempt}")
                print(f"Результат: {best_count}/10")
                print("Промпт:\n", best_prompt, flush=True)
                print("Текст:\n", best_text, flush=True)

            if result["success"]:
                success = True

        if success:
            break

if attempts_stats and len(attempts_stats) % STATS_EVERY != 0:
    print_stats(attempts_stats)
print("\n" + "=" * 50)
print("ПЕРЕБОР ЗАВЕРШЁН")
print(f"Успех: {success}")
print(f"Лучший результат: {best_count}/10")
print(f"Время: {(time.time() - start_all) / 60:.1f} мин")
print("Лучший промпт:\n", best_prompt)
print("Лучший текст:\n", best_text)
print("Лучшие параметры:", best_params)


gpu_count: 4


Перебор:   0%|          | 0/3750 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



Попытка 2 | GPU 1 | 73.6 сек | Промпт №2 | temperature=0.7, top_k=50

Найдено: 1/10 — было
Ещё не найдены: является, стало, стала, была, становится, считается, остается, будет, стал

Попытка 1 | GPU 0 | 78.1 сек | Промпт №3 | temperature=0.8, top_k=100

Найдено: 2/10 — было → была
Ещё не найдены: является, стало, стала, становится, считается, остается, будет, стал

🏆 НОВЫЙ РЕКОРД!
Попытка: 1
Результат: 2/10
Промпт:
 Молодой исследователь отправился в далёкую экспедицию. В своём
    блокноте он фиксировал наблюдения, записывал имена участников,
    описывал открытия и неожиданные происшествия. Однажды вечером
    он сделал следующую запись:
Текст:
  «Нашёл след молодого существа». Учёный шёл по следу несколько дней - тщетно! Наконец его осенило... Он взял перо за кончик левой руки (левая рука была без перчатки) правой рукой принялся водить им вокруг себя на расстоянии нескольких метров от цели эксперимента… И вдруг почувствовал удар электрического тока под кожей лба!.. Профессор замер 

In [ ]:
import json
from datetime import datetime

save_data = {
    "success": success,
    "best_count": best_count,
    "total_words": len(words),
    "found_words": find_progress(best_text)[0],
    "missing_word": find_progress(best_text)[1],
    "prompt": best_prompt,
    "text": best_text,
    "params": best_params,
    "TOTAL_ATTEMPTS": TOTAL_ATTEMPTS,
    "elapsed_minutes": round((time.time() - start_all) / 60, 2),
    "saved_at": datetime.now().isoformat(timespec="seconds")
}

filename = "best_generation.json"

with open(filename, "w", encoding="utf-8") as f:
    json.dump(save_data, f, ensure_ascii=False, indent=4)

print(f"\nРезультат сохранён в файл: {filename}")